In [1]:
import argparse
import json
import os 
import re
import shutil # handling files and folders
import sys
import cv2
import yaml

### HANDLING DIRECTORIES

In [2]:
HERE = os.getcwd()
PIPELINE_ROOT= os.path.dirname(os.path.dirname(HERE))
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(PIPELINE_ROOT)))
CALIB=os.path.join(os.path.dirname(REPO_ROOT),"calibration/calibration_Matries")
ONBOARD_ROOT= os.path.join((REPO_ROOT),"qcar_onboard")
DST=os.path.join(os.path.dirname(PIPELINE_ROOT),"dataset/CoopFront")
print(f"Path of calibration:{CALIB}\nPath where is data on board:{ONBOARD_ROOT}\nPath where the dataset will be placed:{DST}")

Path of calibration:/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/calibration/calibration_Matries
Path where is data on board:/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_onboard
Path where the dataset will be placed:/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/dataset/CoopFront


### CAR REGISTRATION  

In [3]:
DST1=os.path.join(ONBOARD_ROOT,"Inference/192.168.1.198/dataset")
MSK1=os.path.join(os.path.dirname(PIPELINE_ROOT),"shared/visibility_masks/qcar52775_bev_visibility.png")
MSK2=os.path.join(os.path.dirname(PIPELINE_ROOT),"shared/visibility_masks/qcar52776_bev_visibility.png")
DST2=os.path.join(ONBOARD_ROOT,"Inference/192.168.1.158/dataset")
AGENTS={"A1": {"path": DST1, "mask":MSK1}, 
        "A2": {"path": DST2, "mask":MSK2} 
       }
print(AGENTS["A1"]["mask"])
print(AGENTS["A2"]["mask"])


/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/shared/visibility_masks/qcar52775_bev_visibility.png
/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/shared/visibility_masks/qcar52776_bev_visibility.png


### LOADING CALIBRATION

In [4]:
import numpy as np
Path_AG1_Calib= os.path.join(CALIB,"qcar52775/latest_front/qcar52775_front_intrinsics_verified.npz")
Path_AG2_Calib= os.path.join(CALIB,"qcar52776/latest_front/qcar52776_front_intrinsics_verified.npz") 
Cal_AG1=np.load(Path_AG1_Calib)
Cal_AG2=np.load(Path_AG2_Calib)
print(Cal_AG1.files)
print(f"Agent 1 Focal : {Cal_AG1['front_K']}")
print(f"Agent 1 Distortion: {Cal_AG1['front_D']}")
print(f"Agent 1 Distortion: {Cal_AG1['image_size']}")

['front_K', 'front_D', 'image_size']
Agent 1 Focal : [[319.55272432   0.         321.40373768]
 [  0.         317.82473381 243.31500589]
 [  0.           0.           1.        ]]
Agent 1 Distortion: [-0.04116266  0.0033232  -0.00753685  0.00208964]
Agent 1 Distortion: [640 480]


front_K → camera intrinsic matrix: focal lengths + optical center.

front_D → fisheye distortion coefficients.

image_size → image resolution, e.g. width and height.

In [5]:
AGENTS["A1"].update({
    "K": Cal_AG1["front_K"],
    "D": Cal_AG1["front_D"],
    "car": "qcar52775",
    "IP": "192.168.1.198"
})
AGENTS["A2"]["K"], AGENTS["A2"]["D"],AGENTS["A2"]["car"],AGENTS["A2"]["IP"] =Cal_AG2['front_K'],Cal_AG2['front_D'],"qcar52776","192.168.1.158" 
print(AGENTS)

{'A1': {'path': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_onboard/Inference/192.168.1.198/dataset', 'mask': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/shared/visibility_masks/qcar52775_bev_visibility.png', 'K': array([[319.55272432,   0.        , 321.40373768],
       [  0.        , 317.82473381, 243.31500589],
       [  0.        ,   0.        ,   1.        ]]), 'D': array([-0.04116266,  0.0033232 , -0.00753685,  0.00208964]), 'car': 'qcar52775', 'IP': '192.168.1.198'}, 'A2': {'path': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_onboard/Inference/192.168.1.158/dataset', 'mask': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/shared/visibility_masks/qcar52776_bev_visibility.png', 'K': array([[309.78863923,   0.        , 323.44739146],
       [  0.        , 308.55918812, 230.08122271],
       [  0.        ,   0.        ,   1.        ]]), 'D'

### Car Physic Constants and Frame-pairing tolerance

In [6]:
SCALE=10.0 #given that heal expect real vehicles and we are using scale 1/10
PAIR_TOLERANCE_SEC=0.04 #40 ms of tolerance to pair frames
QCAR_L, QCAR_W, QCAR_H = 0.425, 0.192, 0.190  #Length , width , height of the phyiscal car in meters

### Marker-to-physical-center offsets

OFFSET_FWD_M → forward distance from the Vicon marker position to the car’s physical center.

OFFSET_LAT_M → lateral/sideways distance from the Vicon marker to the physical center.

In [7]:
#Extracted from physical measurements and validated with software
OFFSET_FWD_M=0.0343 # this is in meters
OFFSET_LAT_M=-0.0209 # this is in meters

### AGENTS_IDS & Front-camera mount position

In [8]:
FRONT_MOUNT_XYZ = (0.1930, 0.0, 0.0953)  # nominal camera position in the Qcar body frame, meters.
R_CB_NOMINAL= np.array([[0,0,1],[-1,0,0],[0,-1,0]]) #nominal camera postion in the Qcar rotation
AGENT_IDS=tuple(AGENTS.keys())
AGENTS["A1"].update({
    "rotation": R_CB_NOMINAL,
    "translation": np.array(FRONT_MOUNT_XYZ)
})
AGENTS["A2"].update({
    "rotation": R_CB_NOMINAL,
    "translation": np.array(FRONT_MOUNT_XYZ)
})
print(AGENTS)

{'A1': {'path': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_onboard/Inference/192.168.1.198/dataset', 'mask': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/shared/visibility_masks/qcar52775_bev_visibility.png', 'K': array([[319.55272432,   0.        , 321.40373768],
       [  0.        , 317.82473381, 243.31500589],
       [  0.        ,   0.        ,   1.        ]]), 'D': array([-0.04116266,  0.0033232 , -0.00753685,  0.00208964]), 'car': 'qcar52775', 'IP': '192.168.1.198', 'rotation': array([[ 0,  0,  1],
       [-1,  0,  0],
       [ 0, -1,  0]]), 'translation': array([0.193 , 0.    , 0.0953])}, 'A2': {'path': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_onboard/Inference/192.168.1.158/dataset', 'mask': '/mnt/mainvolume/Backup/Projects/HEAL/qcar_testbed_integration/data/qcar_dataset/pipeline/shared/visibility_masks/qcar52776_bev_visibility.png', 'K': array([[309.78863923,   0.       

### IMAGE SIZE & Discovery Trajectories

**Trajectory INdex:** Find a name that starts with `converted_`, contains any text after it, and ends with `_` 
followed by one or more digits; capture those digits.


In [55]:
IMG_W,IMG_H= 640, 480 ### the model expects the image in this resolution 
CAM_PARAMS=AGENTS #putting the whole dictionary AGENTS inside a variable
_TRAJ_INDEX_RE = re.compile(r"^converted_.+_(\d+)$") #index to know which trajectory is
VAL_PAIRS={"10"} #every pair # numbero of experiments
def discover_trajectories(cav_id):
    root=CAM_PARAMS[cav_id]["path"]
    found={}
    for name in os.listdir(root):
        m=_TRAJ_INDEX_RE.match(name) 
        if m and os.path.isdir(os.path.join(root,name)):
            found[m.group(1)]=name
    return found
discover_trajectories("A1")
per_agent = {aid: set(discover_trajectories(aid).keys())
    for aid in AGENT_IDS}
#per_agent.values()
#common=set.intersection(*per_agent.values())
#common=sorted(common)

{'A1': {'10'}, 'A2': {'10'}}


In [57]:
def common_indices():
    per_agent = {aid: set(discover_trajectories(aid).keys())
    for aid in AGENT_IDS}
    if not per_agent or any (not s for s in per_agent): # if any agent dont have trajectories return nothing.
        return[]
    common=set.intersection(*per_agent.values()) # Keeps only the trajectory indices taht exist in all agentes
    print(common)
common_indices()
    

{'10'}
